# 上下文工程
- Compaction: 压缩整合，适合需要长对话连续性的任务，强调上下文的“接力”。
- Structured note-taking: 结构化笔记，适合有里程碑/阶段性成果的迭代式开发与研究。
- Sub-agent architectures: 子代理架构，适合复杂研究与分析，能从并行探索中获益。

## ContextBuilder
- 统一入口：GSSC(Gather-Select-Structure-Compress)流水线。
- 稳定形态：输出固定骨架的上下文模板，便于调试、A/B 测试与评估。采用了分区组织的模板结构：
    - [Role & Policies]：明确 Agent 的角色定位和行为准则
    - [Task]：当前需要完成的具体任务
    - [State]：Agent 的当前状态和上下文信息
    - [Evidence]：从外部知识库检索的证据信息
    - [Context]：历史对话和相关记忆
    - [Output]：期望的输出格式和要求
- 预算考虑：在token预算内尽量保留高价值信息，对超限上下文压缩兜底。
- 最小原则：不引入来源/优先级等分类维度，避免复杂度增长。实践表明，基于相关性和新近性的简单评分机制，在大多数场景下已经足够有效。

### 两个核心数据结构
- **ContextPacket**：候选信息包。每个候选信息都会被封装为一个 ContextPacket，包含内容、时间戳、token 数量和相关性分数等核心属性。这种统一的数据结构简化了后续的选择和排序逻辑。
- **ContextConfig**：配置管理。封装了所有可配置的参数，使得系统行为可以灵活调整。

In [4]:
from dataclasses import dataclass
from typing import Optional, Dict, Any, List
from datetime import datetime
from hello_agents.core.message import Message

In [5]:
@dataclass
class ContextPacket:
    """
    候选信息包
    attrs:
        content: 信息内容
        timestamp: 时间戳
        token_count: token 数量
        relevance_score: 相似性分数0.0-1.0
        metadata: 元数据
    """
    content: str
    timestamp: datetime
    token_count: int
    relevance_score: float = 0.5
    metadata: Optional[Dict[str, Any]] = None

    def __post_init__(self):
        """初始化后处理"""
        if self.metadata is None:
            self.metadata = {}
        # 确保相似性分数在有效范围内
        self.relevance_score = max(0.0, min(1.0, self.relevance_score))

In [6]:
@dataclass
class ContextConfig:
    """
    上下文构建配置
    attrs:
        max_tokens: 最大token数量
        reserve_ratio: 为系统指令预留的比例，0.0-1.0
        min_relevance: 最低相关性阈值
        enable_compression: 是否启用压缩
        recency_weight: 新近性权重，0.0-1.0
        relevance_weight: 相关性权重，0.0-1.0
    """
    max_tokens: int = 100
    reserve_ratio: float = 0.2
    min_relevance: float = 0.1
    enable_compression: bool = True
    recency_weight: float = 0.3
    relevance_weight: float = 0.7

    def __post_init__(self):
        """验证配置参数"""
        assert 0.0 <= self.reserve_ratio <= 1.0, "reserve_ratio must be between 0.0 and 1.0"
        assert 0.0 <= self.min_relevance <= 1.0, "min_relevance must be between 0.0 and 1.0"
        assert abs(self.recency_weight + self.relevance_weight - 1.0) <= 1e-6, "recency_weight + relevance_weight must be equal to 1.0"

### ContextBuilder 的核心是 **GSSC**(Gather-Select-Structure-Compress)流水线
- **gather**：多源信息汇集。从多个来源汇集候选信息，关键在于容错性和灵活性。
    - 容错：外部数据源的调用都被try-except包裹，确保单个源失败不会影响整体流程；
    - 优先级处理：系统指令被标记为高优先级，确保始终被保留；
    - 历史限制：对话历史只保留最近的几条，避免占用上下文窗口；
- **select**：智能信息选择。根据相关性和新近性对候选信息进行评分和选择，直接决定了最终上下文的质量。
    - 评分机制：采用相关性和新近性的加权组合，权重可配置；
    - 贪心算法：按分数从高到低填充，确保在有限预算内选择最有价值的信息；
    - 过滤机制：通过 min_relevance 参数过滤低质量信息；
- **structure**：结构化输出。组织成结构化的context template。
    - 可读性：清晰的分区让人类和模型都更容易理解上下文结构；
    - 可调试性：问题定位更容易，可以快速识别哪个区域的信息有问题；
    - 可扩展性：添加新的信息源只需要创建新的分区；
- **compress**：兜底压缩。对超限上下文进行压缩处理。

In [ ]:
def _gather(
        self,
        user_query: str,
        conversation_history: Optional[List[Message]] = None,
        system_instructions: Optional[str] = None,
        custom_packets: Optional[List[ContextPacket]] = None,
) -> List[ContextPacket]:
    """
    汇集所有候选信息

    Args:
        user_query: 用户查询
        conversation_history: 对话历史
        system_instructions: 系统指令
        custom_packets: 自定义信息包

    Returns:
        List[ContextPacket]: 候选信息列表
    """
    packets = []

    # 1. 添加系统指令（最高优先级，不参与评分）
    if system_instructions:
        packets.append(ContextPacket(
            content=system_instructions,
            timestamp=datetime.now(),
            token_count=self._count_tokens(system_instructions),
            relevance_score=1.0,    # 保证系统指令始终保留
            metadata={'type': 'system_instruction', 'priority': 'high'},
        ))

    # 2. 从记忆系统检索相关记忆
    if self.memory_tool:
        try:
            memory_results = self.memory_tool.run({
                'action': 'search',
                'query': user_query,
                'limit': 10,
                'min_importance': 0.3
            })
            # 解析记忆结果并转换为 ContextPacket
            memory_packets = self._parse_memory_results(memory_results, user_query)
            packets.append(memory_packets)
        except Exception as e:
            print(f"[WARNING] 记忆检索失败: {e}")

    # 3. 从 RAG 系统检索相关知识
    if self.rag_tool:
        try:
            rag_results = self.rag_tool.run({
                "action": "search",
                "query": user_query,
                "limit": 5,
                "min_score": 0.3
            })
            # 解析 RAG 结果并转换为 ContextPacket
            rag_packets = self._parse_rag_results(rag_results, user_query)
            packets.extend(rag_packets)
        except Exception as e:
            print(f"[WARNING] RAG 检索失败: {e}")

    # 4. 添加对话历史（仅保留最近的 N 条）
    if conversation_history:
        recent_history = conversation_history[-5:]  # 默认保留最近5条
        for msg in recent_history:
            packets.append(ContextPacket(
                content=f"{msg.role}: {msg.content}",
                timestamp=msg.timestamp if hasattr(msg, 'timestamp') else datetime.now(),
                token_count=self._count_tokens(msg.content),
                relevance_score=0.6,    # 历史消息的基础相关性
                metadata={'type': 'conversation_history', 'role': msg.role},
            ))

    # 5. 添加自定义信息包
    if custom_packets:
        packets.extend(custom_packets)

    print(f"[ContextBuilder] 汇集了 {len(packets)} 个候选信息包")
    return packets

In [ ]:
def _select(
        self,
        packets: List[ContextPacket],
        user_query: str,
        available_tokens: int
) -> List[ContextPacket]:
    """
    选择最相关的信息包

    Args:
        packets: 候选信息包列表
        user_query: 用户查询，用于计算相关性
        available_tokens: 可用的token数量

    returns：
        List[ContextPacket]: 选中的信息包
    """
    # 1. 分离系统指令和其他信息
    system_packets = [p for p in packets if p.metadata.get('type') == 'system_instruction']
    other_packets = [p for p in packets if p.metadata.get('type') != 'system_instruction']

    # 2. 计算系统指令占用的token
    system_tokens = sum(p.token_count for p in system_packets)
    remaining_tokens = available_tokens - system_tokens

    if remaining_tokens <= 0:
        print(f'[WARNING] 系统指令已占满所有 token 预算')
        return system_packets

    # 3. 为其他信息计算综合分数
    scored_packets = []
    for packet in other_packets:
        # 计算相关性分数
        if packet.relevance_score == 0.5:   # 如果还是默认值0.5，则需要重新计算
            relevance = self._calculate_relevance(packet.content, user_query)
            packet.relevance_score = relevance

        # 计算新近性分数
        recency = self._calculate_recency(packet.timestamp)

        # 综合分数 = 相关性权重 * 相关性分数 + 新近性权重 * 新近性分数
        combined_score = (
            self.config.relevance_weight * packet.relevance_score +
            self.config.recency_weight * recency
        )

        # 过滤低于最小相关性阈值的信息
        if packet.relevance_score >= self.config.min_relevance:
            scored_packets.append((combined_score, packet))

    # 4. 按分数降序排序
    scored_packets.sort(key=lambda x: x[0], reverse=True)

    # 5. 贪心选择：按分数从高到低填充，直到达到 token上限
    selected = system_packets.copy()
    current_tokens = system_tokens

    for score, packet in scored_packets:
        if current_tokens + packet.token_count <= available_tokens:
            selected.append(packet)
            current_tokens += packet.token_count
        else:
            # token 预算已满，停止填充
            break

    print(f"[ContextBuilder] 选择了 {len(selected)} 个信息包，共 {current_tokens} tokens")
    return selected


def _calculate_relevance(content: str, user_query: str) -> float:
    """
    计算内容与查询的相关性：使用简单的关键词重叠算法（可替换为向量相似度计算）

    Args:
        content: 内容文本
        user_query: 用户查询文本

    Returns:
        float: 相似性分数，0.0-1.0
    """
    # 分词，简单实现（可以替换为更复杂的分词器）
    content_words = set(content.lower().split())
    query_words = set(user_query.lower().split())

    if not query_words:
        return 0.0

    # Jaccarb 相似度
    intersection = content_words & query_words
    union = content_words | query_words
    return len(intersection) / len(union) if union else 0.0


def _calculate_recency(timestamp: datetime) -> float:
    """
    时间近因性分数：使用指数衰减模型，24小时内保持高分，之后逐渐衰减

    Args:
        timestamp: 信息的时间戳

    Returns:
        float: 新近性分数，0.0-1.0
    """
    import math

    age_hours = (datetime.now() - timestamp).total_seconds() / 3600

    # 指数衰减
    decay_factor = 0.1  # 衰减系数
    recency_score = math.exp(-decay_factor * age_hours / 24)
    return max(0.0, min(1.0, recency_score))    # 限制在0.0-1.0范围内

In [ ]:
def _structure(
        self,
        selected_packets: List[ContextPacket],
        user_query: str,
) -> str:
    """
    将选中的信息包组成结构化的上下文模板

    Args:
        selected_packets: 选中的信息包列表
        user_query: 用户查询

    Returns:
        str: 结构化的上下文字符串
    """
    # 按类型分组
    system_instructions = []
    evidence = []
    context = []

    for packet in selected_packets:
        packet_type = packet.metadata.get("type", "general")

        if packet_type == 'system_instruction':
            system_instructions.append(packet.content)
        elif packet_type in ['rag_result', 'knowledge']:
            evidence.append(packet.content)
        else:
            context.append(packet)

    # 构建结构化模板
    sections = []

    # [Role & Policies]
    if system_instructions:
        sections.append("[Role & Policies]\n" + "\n".join(system_instructions))

    # [Task]
    sections.append("[Task]\n" + "\n".join(user_query))

    # [Evidence]
    if evidence:
        sections.append("[Evidence]\n" + "\n---\n".join(evidence))

    # [Context]
    if context:
        sections.append("[Context]\n" + "\n".join(context))

    # [Output]
    sections.append("[Output]\n请基于以上信息，提供准确、有据的回答。")

    return "\n\n".join(sections)

In [ ]:
def _compress(
        self,
        context: str,
        max_tokens: int
) -> str:
    """
    压缩超限的上下文：简单贪心顺序填充
    Args:
        context：原始上下文
        max_tokens：最大token限制
    Returns：
        str：压缩后的上下文
    """
    current_tokens = self._count_tokens(context)

    if current_tokens < max_tokens:
        return context  # 无需压缩

    print(f"[ContextBuilder] 上下文超限({current_tokens} > {max_tokens}), 执行压缩")

    # 分区压缩：保持结构完整性
    sections = context.split("\n\n")
    compressed_sections = []
    current_total = 0

    for section in sections:
        section_tokens = self._count_tokens(section)

        if current_total + section_tokens <= max_tokens:
            # 完整保留
            compressed_sections.append(section)
            current_total += section_tokens
        else:
            # 部分保留
            remaining_tokens = max_tokens - current_total
            if remaining_tokens > 50:   # 至少保留50tokens
                # 简单截断，实际可以使用 LLM 摘要
                truncated = self._truncate_text(section, remaining_tokens)
                compressed_sections.append(truncated + "\n[...内容已压缩...]")
            break

    compressed_context = "\n\n".join(compressed_sections)
    final_tokens = self._count_tokens(compressed_context)
    print(f"[ContextBuilder] 压缩完成：{current_tokens} -> {final_tokens} tokens")
    return compressed_context


def _truncate_text(self, text: str, max_tokens: int) -> str:
    """
    截断文本到指定的token数量
    Args：
        text：原始文本
        max_tokens：最大token数量
    Returns:
        str：截断后的文本
    """
    # 简单实现：按字符比例估算，实际应该使用精确的 tokenizer
    char_per_token = len(text) / self._count_tokens(text) if self._count_tokens(text) > 0 else 4
    max_chars = int(char_per_token * max_tokens)
    return text[:max_chars]


def _count_tokens(self, text: str) -> int:
    """估算文本的 token 数量

    Args:
        text: 文本内容

    Returns:
        int: token 数量
    """
    # 简单估算：中文 1 字符 ≈ 1 token，英文 1 单词 ≈ 1.3 tokens，实际应该用 tokenizer
    chinese_chars = sum(1 for ch in text if '\u4e00' <= ch <= '\u9fff')
    english_words = len([w for w in text.split() if w])
    return int(chinese_chars + english_words * 1.3)

## NoteTool: 结构化笔记, Structured Note-taking

- 以 Markdown 作为载体，开头使用 YAML 前置元数据记录关键信息，正文用于记录状态、结论、阻塞、行动项等内容
- 适合：
    - 长期项目追踪：记录 task_state当前阶段的任务状态和进度、conclusion每个阶段结束后的关键结论、blocker遇到的问题的阻塞点、action下一步的行动计划
    - 研究任务管理：整理文档时记录 conclusion每篇的核心观点、action待深入研究的主题、reference重要参考文献
    - 与ContextBuilder配合：每轮对话前，agent通过search或list操作检索相关笔记，并将其注入到上下文中

### 存储格式
- (1)笔记文件格式：每一个笔记都是一个独立的 .md 文件
```
---
id: note_20250119_153000_0
title: 项目进展 - 第一阶段
type: task_state
tags: [refactoring, phase1, backend]
created_at: 2025-01-19T15:30:00
updated_at: 2025-01-19T15:30:00
---

# 项目进展 - 第一阶段

## 完成情况

已完成数据模型层的重构,主要改动包括:

1. 统一了实体类的命名规范
2. 引入了类型提示,提升代码可维护性
3. 优化了数据库查询性能

## 测试覆盖

- 单元测试覆盖率: 85%
- 集成测试覆盖率: 70%

## 下一步计划

1. 重构业务逻辑层
2. 解决依赖冲突问题
3. 提升集成测试覆盖率至85%
```

- (2)索引文件：NoteTool维护一个 note_index.json 文件，用于快速检索和管理笔记
```
{
  "note_20250119_153000_0": {
    "id": "note_20250119_153000_0",
    "title": "项目进展 - 第一阶段",
    "type": "task_state",
    "tags": ["refactoring", "phase1", "backend"],
    "created_at": "2025-01-19T15:30:00",
    "updated_at": "2025-01-19T15:30:00",
    "file_path": "./notes/note_20250119_153000_0.md"
  }
}
```

NoteTool核心操作：
- create：创建笔记
- read：读取笔记
- update：更新笔记
- search：搜索笔记
- list：列出笔记
- summary：笔记摘要
- delete：删除笔记